#Uploads - Bronze
1. Set up w/
    - Tables/hnm_articles_bronze (HuggingFace)
    - Tables/hnm_transactions_bronze (master transactions table)
2. Manual hnm_weekly_transactions uplaod
3. Hypothetically hnm_weekly_transactions Azure API call


Setup & Config

In [0]:
import os
from delta.tables import DeltaTable
import re
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, DateType, StringType, IntegerType, DoubleType)


In [0]:
# File locations

# Main transaction Table
transactions = "workspace.default.hnm_transactions_bronze"

# Weekly uploads Volume
input_dir = "/Volumes/workspace/default/hnm_weekly_upload/"

Inspect main table date range

In [0]:
transactions_table = spark.table(transactions)

summary = transactions_table.select(
    F.date_format(F.min("t_dat"), "yyyy-MM-dd (EEEE)").alias("min_formatted"),
    F.date_format(F.max("t_dat"), "yyyy-MM-dd (EEEE)").alias("max_formatted"),
).collect()[0]

print(f"Dataset range = {summary['min_formatted']} to {summary['max_formatted']}"
      f"\n{transactions_table.count():,} rows in main transactions table")

Dataset range = 2018-09-20 (Thursday) to 2020-08-21 (Friday)
30,597,413 rows in main transactions table


Load last weekly transactions file from volumes.


In [0]:

# Find the newest weekly upload in the (hnm_weekly_upload) input directory.
# The file name holds the week dates (week_35_2020-08-24_to_2020-08-30.csv), so "newest" = latest start date in the name.
files = [f for f in dbutils.fs.ls(input_dir) if f.name.lower().endswith(".csv")]

if len(files) == 0:
    raise ValueError(f"No CSV files found in {input_dir}")

def week_start(f):
    match = re.search(r"\d{4}-\d{2}-\d{2}", f.name)
    return match.group(0) if match else ""      # a file with no date in its name counts as the oldest

newest = max(files, key=lambda f: (week_start(f), f.name))
input_path = newest.path
print(f"Using {newest.name} (newest of {len(files)} CSV files)")

weekly_transactions = (
    spark.read
        .option("header", True)
        .csv(input_path)
)

summary = weekly_transactions.select(
    F.date_format(F.min("t_dat"), "yyyy-MM-dd (EEEE)").alias("min_formatted"),
    F.date_format(F.max("t_dat"), "yyyy-MM-dd (EEEE)").alias("max_formatted"),
).collect()[0]

print(f"Dataset range = {summary['min_formatted']} to {summary['max_formatted']}"
      f"\n{weekly_transactions.count():,} rows in last weekly_transactions table")

Dataset range = 2020-08-24 (Monday) to 2020-08-30 (Sunday)
286,525 rows in last weekly_transactions table


Aggregate weekly transaction if max date isn't newer than transactions master table.


In [0]:
# Double check: guard against stale / already-ingested weekly files

# The weekly file's max date must be newer than the max date already in the main table. 
# This catches both re-uploading an old file and re-running the notebook twice on the same file.

weekly_max_dat = weekly_transactions.agg(
    F.max(F.to_date(F.col("t_dat"))).alias("max_dat")
    ).collect()[0]["max_dat"]

if weekly_max_dat is None:
    raise ValueError(f"Could not parse any dates from t_dat in {input_path}.")

should_append = True

if spark.catalog.tableExists(transactions):
    existing_max_dat = spark.sql(
        f"""
        SELECT MAX(to_date(t_dat)) AS max_dat
        FROM {transactions}
        """
    ).collect()[0]["max_dat"]

    if (existing_max_dat is not None and weekly_max_dat <= existing_max_dat):
        print(f"No new weekly data, using existing {transactions} through {existing_max_dat}.")
        should_append = False

if should_append:
    (weekly_transactions
        .write.format("delta")
        .mode("append").saveAsTable(transactions))
    print(f"New weekly data appended to {transactions}.")

else:
    print(f"Skipped ingestion. Existing {transactions} will be used by downstream pipeline stages.")

New weekly data appended to workspace.default.hnm_transactions_bronze.


In [0]:
# Quick Validation 

spark.sql(f"""
SELECT
    MIN(t_dat) AS min_transaction_date,
    MAX(t_dat) AS max_transaction_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT article_id) AS articles
FROM {transactions}
""").show()

+--------------------+--------------------+----------+---------+--------+
|min_transaction_date|max_transaction_date|total_rows|customers|articles|
+--------------------+--------------------+----------+---------+--------+
|          2018-09-20|          2020-08-30|  30883938|  1342770|  101696|
+--------------------+--------------------+----------+---------+--------+

